<a href="https://colab.research.google.com/github/moise97/Extract_-_Structure_Data_from_SDFs_pharmaceutical_documentation/blob/main/Route_Queries_Across_Multiple_PDF_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate sentencepiece pdfplumber pandas
print('✅ Packages ready.')

In [ ]:
from google.colab import files
import pdfplumber
import uuid

print('📂 Upload your PDF...')
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {pdf_filename}')

# --------------------------------------------------------------------------
# Build pdf_metadata_store — one dict per page
# This is the structure all downstream functions operate on
# --------------------------------------------------------------------------
FILE_ID  = str(uuid.uuid4())   # unique ID for this file
USER_ID  = 'xyz'               # ✏️ change to your user ID if needed
YEAR     = '2024'              # ✏️ change to match your document year

pdf_metadata_store = []

with pdfplumber.open(pdf_filename) as pdf:
    for i, page in enumerate(pdf.pages):
        text = (page.extract_text() or '').strip()
        pdf_metadata_store.append({
            'file_id':     FILE_ID,
            'user_id':     USER_ID,
            'doc_type':    None,          # filled in Cell 5
            'year':        YEAR,
            'filename':    pdf_filename,
            'page_number': i + 1,
            'text':        text
        })

print(f'\n✅ Built pdf_metadata_store with {len(pdf_metadata_store)} page(s)')
print(f'   file_id : {FILE_ID}')
print(f'   Preview page 1: {pdf_metadata_store[0]["text"][:120]}...')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print(f'Loading {MODEL_NAME}...\n')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
llm_model.eval()

print('✅ TinyLlama loaded!')
if torch.cuda.is_available():
    print(f'   Device : {torch.cuda.get_device_name(0)}')
    print(f'   Memory : {torch.cuda.memory_allocated()/1e9:.2f} GB used')

In [ ]:
import re, json

# ── Known document types ────────────────────────────────────────────────────
KNOWN_DOC_TYPES = [
    'cover_letter',
    'certificate_of_quality',
    'packaging_specification',
    'bse_tse_declaration',
    'material_description',
    'supplier_qualification',
    'chain_of_custody',
    'unknown'
]
TYPES_STR = ', '.join(KNOWN_DOC_TYPES)

# ── Keyword map for fallback search ─────────────────────────────────────────
KEYWORD_MAP = {
    'cover_letter':            ['cover letter', 'dear', 'enclosed', 'attached'],
    'certificate_of_quality':  ['certificate of quality', 'lot number', 'article number',
                                 'sterilization', 'batch', 'expiry', 'quality'],
    'packaging_specification': ['packaging', 'carton', 'dimensions', 'weight',
                                 'units per box', 'specification'],
    'bse_tse_declaration':     ['bse', 'tse', 'bovine', 'spongiform', 'declaration',
                                 'animal origin'],
    'material_description':    ['material', 'composition', 'polymer', 'grade',
                                 'raw material', 'substance'],
    'supplier_qualification':  ['supplier', 'vendor', 'qualification', 'approved',
                                 'audit', 'certification'],
    'chain_of_custody':        ['chain of custody', 'custody', 'transfer', 'traceability',
                                 'tracking'],
    'unknown':                 []
}


def ask_tinyllama(prompt: str, max_new_tokens: int = 80) -> str:
    """Send a prompt to TinyLlama and return the response text."""
    chat = [
        {'role': 'system', 'content': 'You are a precise document classification assistant. Follow instructions exactly. Respond only as instructed — no explanation.'},
        {'role': 'user',   'content': prompt}
    ]
    formatted = tokenizer.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt').to(llm_model.device)
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_doc_type(raw: str) -> str:
    """Extract a valid doc type from raw LLM output. Returns 'unknown' on failure."""
    # Try JSON first
    try:
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            candidate = str(parsed.get('doc_type', '')).strip().lower()
            if candidate in KNOWN_DOC_TYPES:
                return candidate
    except Exception:
        pass

    # Scan raw text for any known type string
    raw_lower = raw.lower()
    for t in KNOWN_DOC_TYPES:
        if t in raw_lower:
            return t

    return 'unknown'


print('✅ Helper functions ready.')
print(f'   Known types: {KNOWN_DOC_TYPES}')

In [ ]:
def classify_query_llm(query: str) -> str:
    """
    Classifies a user query into one of the known document types.
    Returns predicted_doc_type as a string.
    """
    prompt = f"""A user has submitted the following query about a document:

Query: "{query}"

Based on the query, which type of document is the user most likely asking about?

You MUST choose exactly one from this list:
{TYPES_STR}

Rules:
- If the query mentions part numbers, article numbers, lot numbers, or quality → certificate_of_quality
- If the query mentions BSE, TSE, bovine, animal origin → bse_tse_declaration
- If the query mentions packaging, dimensions, carton, box → packaging_specification
- If the query mentions material, composition, polymer → material_description
- If the query mentions supplier, vendor, audit → supplier_qualification
- If the query mentions chain of custody, traceability, tracking → chain_of_custody
- If the query is a general introduction or transmittal → cover_letter
- If none of the above apply → unknown

Respond ONLY with this JSON:
{{"doc_type": "<type_from_list>"}}

JSON:"""

    raw = ask_tinyllama(prompt, max_new_tokens=60)
    result = extract_doc_type(raw)
    return result


# ── Test it ──────────────────────────────────────────────────────────────────
# ✏️ Change this query to test different inputs
USER_QUERY = 'What are the part numbers listed in this document?'

print(f'Query: "{USER_QUERY}"')
print('Classifying...')

predicted_doc_type = classify_query_llm(USER_QUERY)

print(f'\n✅ predicted_doc_type = "{predicted_doc_type}"')

In [ ]:
import time

def classify_doc_type_llm(page_text: str) -> str:
    """
    Classifies a single page's text into one of the known document types.
    Returns a doc_type string.
    """
    snippet = page_text[:700] if page_text else '[EMPTY PAGE]'

    prompt = f"""You are classifying a page from a pharmaceutical or medical device document.

PAGE TEXT:
\"\"\"
{snippet}
\"\"\"

Choose exactly one document type from this list:
{TYPES_STR}

Rules:
- certificate of quality / lot / batch / sterilization → certificate_of_quality
- BSE / TSE / bovine / animal origin → bse_tse_declaration
- packaging / carton / dimensions / units per box → packaging_specification
- material / composition / polymer / substance → material_description
- supplier / vendor / audit / qualification → supplier_qualification
- chain of custody / traceability → chain_of_custody
- cover letter / introduction / transmittal → cover_letter
- empty or unrecognisable → unknown

Respond ONLY with this JSON:
{{"doc_type": "<type_from_list>"}}

JSON:"""

    raw = ask_tinyllama(prompt, max_new_tokens=60)
    return extract_doc_type(raw)


# ── Loop through all pages ───────────────────────────────────────────────────
print(f'Classifying {len(pdf_metadata_store)} page(s)...\n')
print('-' * 55)

for page in pdf_metadata_store:
    doc_type = classify_doc_type_llm(page['text'])
    page['doc_type'] = doc_type
    print(f'  Page {page["page_number"]:>3} → "{doc_type}"')
    time.sleep(0.3)   # small delay to avoid GPU thrashing

print('-' * 55)
print(f'\n✅ All pages classified.')

In [ ]:
# Filter to matching pages
matched_documents = [
    page for page in pdf_metadata_store
    if page['doc_type'] == predicted_doc_type
]

print(f'predicted_doc_type : "{predicted_doc_type}"')
print(f'Total pages        : {len(pdf_metadata_store)}')
print(f'Matched pages      : {len(matched_documents)}')

if matched_documents:
    print('\nMatched pages:')
    for m in matched_documents:
        print(f'  Page {m["page_number"]} — {m["text"][:80]}...')
else:
    print('\n⚠️  No direct matches found — fallback will run in Cell 8.')

In [ ]:
def keyword_fallback(doc_type: str, pages: list) -> list:
    """
    Searches all pages for keywords associated with doc_type.
    Returns pages scored by keyword hit count, best match first.
    """
    keywords = KEYWORD_MAP.get(doc_type, [])
    if not keywords:
        print('  No keywords defined for this type — returning all pages.')
        return pages

    scored = []
    for page in pages:
        text_lower = page['text'].lower()
        hits = sum(1 for kw in keywords if kw in text_lower)
        if hits > 0:
            scored.append((hits, page))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [page for _, page in scored]


if not matched_documents:
    print(f'Running keyword fallback for "{predicted_doc_type}"...')
    matched_documents = keyword_fallback(predicted_doc_type, pdf_metadata_store)

    if matched_documents:
        print(f'✅ Fallback found {len(matched_documents)} candidate page(s).')
        for m in matched_documents:
            print(f'  Page {m["page_number"]} — {m["text"][:80]}...')
    else:
        print('⚠️  Fallback also found nothing. matched_documents is empty.')
else:
    print(f'✅ LLM matched {len(matched_documents)} page(s) — fallback not needed.')

In [ ]:
import json
from google.colab import files

# ── Assemble result dict ─────────────────────────────────────────────────────
routing_result = {
    'query':              USER_QUERY,
    'predicted_doc_type': predicted_doc_type,
    'matched_documents':  matched_documents
}

# ── Print it ─────────────────────────────────────────────────────────────────
json_output = json.dumps(routing_result, indent=2)
print('Final Routing Result:')
print('=' * 60)
print(json_output)
print('=' * 60)
print(f'\nQuery              : {routing_result["query"]}')
print(f'Predicted doc type : {routing_result["predicted_doc_type"]}')
print(f'Matched pages      : {len(routing_result["matched_documents"])}')

# ── Save + download ───────────────────────────────────────────────────────────
with open('routing_result.json', 'w') as f:
    f.write(json_output)

print('\n✅ Saved as routing_result.json')
files.download('routing_result.json')

In [ ]:
import pandas as pd

# Full page classification table
df_all = pd.DataFrame(pdf_metadata_store)[['page_number', 'doc_type', 'filename']]
df_all['text_preview'] = [p['text'][:60] + '...' for p in pdf_metadata_store]

print('All pages:')
print(df_all.to_string(index=False))

print(f'\nMatched pages (doc_type = "{predicted_doc_type}"):')
df_matched = pd.DataFrame(matched_documents)[['page_number', 'doc_type', 'filename']]
df_matched['text_preview'] = [m['text'][:60] + '...' for m in matched_documents]
print(df_matched.to_string(index=False))